# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/cgm/cgm.parquet'
df_cgm = read_parquet(pfile)
print(df_cgm.shape)
df_cgm.head()

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/cgm/iglu.parquet'
df_iglu = read_parquet(pfile)
print(df_iglu.shape)
df_iglu.head()

In [ ]:
df_iglu = pd.merge(df_iglu,
                    df_cgm[['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_timestamp', 'timezone']], 
                    on=['participant_id', 'cohort', 'research_stage', 'array_index'])
df_iglu.head()

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/cgm/iglu_daily.parquet'
df_iglud = read_parquet(pfile)
print(df_iglud.shape)
df_iglud.head()

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_iglud)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Data filetering

In [ ]:
df_iglud = df_iglud.sort_values(
    ['participant_id', 'research_stage', 'collection_date']
)

def process_one_collection(g):
    # 如果天数不足 3 天，无法去头去尾，直接丢弃
    if len(g) < 3:
        return None

    # 去掉第一天和最后一天
    g_mid = g.iloc[1:-1]

    # 对数值列求均值
    numeric_mean = g_mid.select_dtypes('number').mean()

    # 取第 2 天的 collection_date
    result = numeric_mean.to_dict()
    result['participant_id'] = g.iloc[0]['participant_id']
    result['cohort'] = g.iloc[0]['cohort']
    result['research_stage'] = g.iloc[0]['research_stage']
    result['collection_date'] = g.iloc[1]['collection_date']

    return pd.Series(result)

df_iglud_agg = (
    df_iglud
    .groupby(['participant_id', 'research_stage'], as_index=False)
    .apply(process_one_collection)
    .dropna(how='all')
    .reset_index(drop=True)
)



In [ ]:
df_iglud_agg = df_iglud_agg.drop(columns=['array_index', 'connection_day'])

# Save

In [ ]:
df_iglud_agg.reset_index(drop=True, inplace=True)
pfile = 'pheno_data/202512/017-cgm_imputed.parquet'
df_iglud_agg.to_parquet(pfile, index=False)

In [ ]:
pfile = 'pheno_data/202512/017-cgm_imputed.parquet'
df_iglud = pd.read_parquet(pfile)
print(df_iglud.shape)
df_iglud.head()